<p><b><h1 style="font-size:30px;text-align: center;">- Data Engineering  - Exercise 9/3 -</h1></b>
<b><h1 style="font-size:25px;text-align: center;">- Pattern / Sample Reduction (Instance Reduction) in Practice -</h1></b></p>
<div style="text-align:center; margin:16px 0 28px;">
<img alt="Algebra Bernays" src="https://futsal-dinamo.hr/wp-content/uploads/2021/01/Algebra-Bernays.png" style="width:600px; max-width:60%; height:auto; margin-top:25px;"/>
</div>
<hr/>
<p><b><em>Made: January 2026.</em> </b></p>
<p><b><em>Author: Adjunct Lecturer Mislav Spajić, M.Eng. (Comp.)</em></b></p>
<hr/>


## Introduction

Feature reduction changes the **number of columns** (features), while pattern/sample (instance) reduction changes the **number of rows** (training samples).
In data engineering pipelines, pattern reduction matters because it can **cut training time**, **lower memory pressure**, and **reduce compute cost** when datasets scale to millions of records.
In this exercise we compare simple, practical instance-reduction strategies on a large synthetic dataset:
- random undersampling
- stratified undersampling
- clustering-based reduction (prototypes)
- outlier removal (indirect reduction)

We only mention (but do not implement) smarter prototype/instance selection methods like **CNN/ENN**, **Tomek links**, etc.


In [ ]:
# Setup

import time
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import IsolationForest

SEED = 42
rng = np.random.default_rng(SEED)

# --- Memory (RSS) helper ---
_PSUTIL_AVAILABLE = True
try:
    import psutil
except Exception:
    _PSUTIL_AVAILABLE = False

def rss_mb() -> float:
    """Process RSS in MB. If psutil is missing, returns NaN."""
    if not _PSUTIL_AVAILABLE:
        return float('nan')
    p = psutil.Process()
    return p.memory_info().rss / (1024 ** 2)

if not _PSUTIL_AVAILABLE:
    print("NOTE: psutil is not installed -> RSS memory metrics will be NaN.")

def class_balance(y: np.ndarray) -> pd.Series:
    s = pd.Series(y).value_counts(normalize=True).sort_index()
    s.index = [f"class_{i}" for i in s.index]
    return s

def build_model() -> Pipeline:
    # Simple and fast baseline model
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SGDClassifier(
            loss="log_loss",
            alpha=1e-4,
            max_iter=5,
            tol=1e-3,
            random_state=SEED,
        )),
    ])

# Will be set after dataset creation
X_test = None
y_test = None

def benchmark(method_name: str, X_train: np.ndarray, y_train: np.ndarray) -> dict:
    """Fit the model and evaluate on the fixed test set."""
    assert X_test is not None and y_test is not None, "Create the dataset first (X_test/y_test)."

    model = build_model()

    rss_before = rss_mb()
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0
    rss_after = rss_mb()

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    row = {
        "method": method_name,
        "n_train": int(len(y_train)),
        "train_time_s": float(train_time),
        "accuracy": float(acc),
        "rss_before_mb": float(rss_before),
        "rss_after_mb": float(rss_after),
        "rss_delta_mb": float(rss_after - rss_before) if (np.isfinite(rss_before) and np.isfinite(rss_after)) else float('nan'),
    }

    print(f"[{method_name}] n_train={row['n_train']:,} | time={row['train_time_s']:.2f}s | acc={row['accuracy']:.4f} | rssΔ={row['rss_delta_mb']:.1f} MB")
    return row


## Generate a dataset with 1,000,000 training rows

- We generate a **binary** classification dataset with **class imbalance (70/30)**.
- We keep a **fixed test set** so every reduction method is evaluated fairly.
- We store features as **float32** to reduce memory.


In [ ]:
N_TRAIN = 1_000_000
N_TEST = 100_000
N_TOTAL = N_TRAIN + N_TEST

X, y = make_classification(
    n_samples=N_TOTAL,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_repeated=0,
    n_clusters_per_class=2,
    weights=[0.7, 0.3],
    flip_y=0.01,
    class_sep=1.0,
    random_state=SEED,
)

X = X.astype(np.float32, copy=False)
y = y.astype(np.int8, copy=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=N_TRAIN,
    test_size=N_TEST,
    stratify=y,
    random_state=SEED,
)

print("X_train:", X_train.shape, X_train.dtype)
print("X_test :", X_test.shape, X_test.dtype)
print("y_train balance:\n", class_balance(y_train))
print("y_test  balance:\n", class_balance(y_test))

approx_mb = (X_train.nbytes + X_test.nbytes + y_train.nbytes + y_test.nbytes) / (1024**2)
print(f"Approx in-memory size of X/y arrays: {approx_mb:,.1f} MB")
print(f"Process RSS now: {rss_mb():.1f} MB")


## Baseline model on full data
We train the same model on the full 1,000,000-row training set to establish a baseline for time, accuracy, and RSS change during training.


In [ ]:
results = []
results.append(benchmark("baseline_full", X_train, y_train))


## Undersampling methods
We keep the test set fixed and train on a **10%** subset of the original training data.


In [ ]:
# Random undersampling (10%)
fraction = 0.10
n_sample = int(len(y_train) * fraction)

idx = rng.choice(len(y_train), size=n_sample, replace=False)
X_rus = X_train[idx]
y_rus = y_train[idx]

print("Random undersample:")
print("  shape:", X_rus.shape)
print("  class balance:\n", class_balance(y_rus))

results.append(benchmark("random_10%", X_rus, y_rus))


In [ ]:
# Stratified undersampling (10%)
idx_all = np.arange(len(y_train))

idx_strat, _ = train_test_split(
    idx_all,
    train_size=n_sample,
    stratify=y_train,
    random_state=SEED,
)

X_sus = X_train[idx_strat]
y_sus = y_train[idx_strat]

print("Stratified undersample:")
print("  shape:", X_sus.shape)
print("  class balance (sample):\n", class_balance(y_sus))
print("  class balance (full)  :\n", class_balance(y_train))

results.append(benchmark("stratified_10%", X_sus, y_sus))


## Clustering-based reduction (prototypes)
Idea: learn **K cluster centers** on a subset, then treat those centers as a compact training set.
We assign each prototype a label by **majority vote** of the points in its cluster (using the clustering subset).


In [ ]:
K = 1000
CLUSTER_SUBSET = 200_000

idx_sub = rng.choice(len(y_train), size=CLUSTER_SUBSET, replace=False)
X_sub = X_train[idx_sub]
y_sub = y_train[idx_sub]

print("Clustering subset:", X_sub.shape)

kmeans = MiniBatchKMeans(
    n_clusters=K,
    batch_size=4096,
    n_init="auto",
    random_state=SEED,
)
t0 = time.perf_counter()
kmeans.fit(X_sub)
print(f"MiniBatchKMeans fit time: {time.perf_counter() - t0:.2f}s")

clusters = kmeans.labels_.astype(np.int32, copy=False)
centers = kmeans.cluster_centers_.astype(np.float32, copy=False)

# Majority vote label per cluster (binary classification -> counts[:,0] vs counts[:,1])
n_classes = int(np.max(y_train)) + 1
counts = np.zeros((K, n_classes), dtype=np.int32)
np.add.at(counts, (clusters, y_sub.astype(np.int32)), 1)

nonempty = counts.sum(axis=1) > 0
y_proto = counts.argmax(axis=1).astype(np.int8)

X_proto = centers[nonempty]
y_proto = y_proto[nonempty]

print("Prototypes:")
print("  K requested:", K)
print("  non-empty clusters:", X_proto.shape[0])
print("  class balance:\n", class_balance(y_proto))

results.append(benchmark(f"kmeans_prototypes_{X_proto.shape[0]}", X_proto, y_proto))


## Outlier removal (indirect reduction)
We use **IsolationForest** to flag outliers and then train only on the inliers.

**Caution:** removing outliers can also remove rare but important cases (e.g., fraud or critical failures).


In [ ]:
ISO_SUBSET = 200_000
idx_iso = rng.choice(len(y_train), size=ISO_SUBSET, replace=False)
X_iso = X_train[idx_iso]

iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=SEED,
    n_jobs=-1,
)

t0 = time.perf_counter()
iso.fit(X_iso)
print(f"IsolationForest fit time (subset): {time.perf_counter() - t0:.2f}s")

def predict_in_chunks(model, X, chunk_size=200_000):
    preds = np.empty((X.shape[0],), dtype=np.int8)
    for start in range(0, X.shape[0], chunk_size):
        end = min(start + chunk_size, X.shape[0])
        preds[start:end] = model.predict(X[start:end]).astype(np.int8)
    return preds

t0 = time.perf_counter()
preds = predict_in_chunks(iso, X_train, chunk_size=200_000)  # 1=inlier, -1=outlier
print(f"Outlier prediction time (train): {time.perf_counter() - t0:.2f}s")

inlier_mask = preds == 1
X_in = X_train[inlier_mask]
y_in = y_train[inlier_mask]

removed = int((~inlier_mask).sum())
print("After outlier removal:")
print(f"  removed outliers: {removed:,} ({removed/len(y_train):.2%})")
print("  remaining shape:", X_in.shape)
print("  class balance:\n", class_balance(y_in))

results.append(benchmark("isoforest_inliers", X_in, y_in))


## Results table
We summarize the impact of each instance reduction method on **training time**, **accuracy**, and **RSS delta**.


In [ ]:
df = pd.DataFrame(results)[["method", "n_train", "train_time_s", "accuracy", "rss_delta_mb"]]
df = df.sort_values("train_time_s", ascending=True).reset_index(drop=True)
df


## Quick discussion
- **Random undersampling** is the fastest to implement, but may throw away informative edge cases.
- **Stratified undersampling** is usually safer for imbalanced classes (keeps label proportions).
- **Clustering prototypes** can compress data aggressively while keeping a "map" of the space, but may lose local detail.
- **Outlier removal** can reduce noise and speed up training, but risks dropping rare-but-important cases.
- Smarter instance selection (e.g., **CNN/ENN**, **Tomek links**) tries to keep "useful" boundary points and remove redundant/noisy ones—often better than naive sampling, but typically more expensive.


## Conclusion

Feature reduction and pattern/sample reduction address different—but complementary—ways of simplifying data.

### Feature reduction (columns)
Feature reduction focuses on decreasing the number of variables (**features**) in a dataset. The goal is to remove irrelevant or redundant information while keeping the most informative attributes.

- **Feature selection** keeps a subset of the original features  
  → better interpretability (you still know what each feature means)
- **Dimensionality reduction** transforms features into a smaller set of new (latent) features  
  → preserves main structure/variance, but the new features may be harder to interpret

### Pattern reduction (rows)
Pattern reduction reduces the number of **instances** (**samples / rows**) in the dataset. Instead of removing variables, it removes redundant, less informative, or noisy points while trying to preserve representativeness.

Common techniques include:
- **Random undersampling**
- **Stratified undersampling** (preserves class proportions)
- **Clustering-based reduction** (representative **prototypes**)
- **Outlier removal** (indirect instance reduction)

### Why this matters in data engineering
In practical workflows, feature and pattern reduction are often combined:
- Feature reduction simplifies the feature space and lowers model complexity.
- Pattern reduction lowers **computational cost**, **memory usage**, and **training time**—especially at large scale.

Understanding the distinction helps data engineers build more efficient, scalable, and resource-aware ML pipelines without unnecessarily sacrificing performance.
